In [39]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import geopandas as gpd

In [5]:
df_20182021 = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_cleaned_2018_2021.csv", thousands=",")
df_20212026 = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_cleaned_2021_2026.csv", thousands=",")

C:\Users\gelet\AppData\Local\Temp\ipykernel_23404\873818448.py:1: DtypeWarning: Columns (32: search_types, 41: legal_basises_relevant_factors, 42: subject_restrained, 43: subject_restraint_devices) have mixed types. Specify dtype option on import or set low_memory=False.
  df_20182021 = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_cleaned_2018_2021.csv", thousands=",")


In [5]:
print(df_20182021.shape)
print(df_20212026.shape)

(148205, 47)
(86729, 47)


In [6]:
# Change time variable into datetime from string
df_20182021["EventDate"] = df_20182021["EventDate"].str.strip()
df_20182021["EventDate"] =  pd.to_datetime(df_20182021["EventDate"], format="%Y %b %d %I:%M:%S %p", errors="coerce")

df_20212026["EventDate"] = df_20212026["EventDate"].str.strip()
df_20212026["EventDate"] =  pd.to_datetime(df_20212026["EventDate"], format="%Y %b %d %I:%M:%S %p", errors="coerce")

In [28]:
# Combine dataframes
df = pd.concat([df_20182021, df_20212026], ignore_index=True)

In [29]:
# Add extra date variables
df["month"] = df["EventDate"].dt.month
df["year"] = df["EventDate"].dt.year
df["weekend"] = (df["EventDate"].dt.weekday >= 5).astype(int)

In [30]:
# Only examine stops on or after March 1, 2018
df = df[(df["month"] >= 3) & (df["year"] >= 2018)]

In [31]:
# Only examine vehicle stops
df = df[(df["subject_type"] == "Passenger") | (df["subject_type"] == "Driver")]

In [32]:
# Recoding some categorical variables
df["SubjectGender"] = df["SubjectGender"].map({"FEMALE": 1, "MALE": 0})
df["SubjectHasPhotoID"] = df["SubjectHasPhotoID"].map({"Yes": 1, "No": 0})
df["isDriver"] = df["subject_type"].map({"Driver":1, "Passenger":0})
df["search_occurred"] = df["search_occurred"].map({"Yes": 1, "No": 0})
df["isLA"] = (df["SubjectDriverLicState"] == "LA").astype("Int64")
df.loc[df["SubjectDriverLicState"].isna(), "isLA"] = pd.NA
df["isTrafficViolation"] = (df["StopDescription"] == "TRAFFIC VIOLATION").astype("Int64")

In [33]:
# Dropping post search variables
# We cannot use these variables as predictors because they may/will only happen after a search already occurs
df = df.drop(["evidence_seized", "search_types", "consent_to_search", "pat_down", "consent_form_completed", "stripbody_cavity_search", "legal_basises", "legal_basises_relevant_factors", "stop_results", "subject_restrained", "subject_restraint_devices", "exit_vehicle", "evidence_types"], axis=1)

# Dropping redundant variables
df = df.drop(["EventDate", "ActionsTaken", "CreatedDateTime", "LastModifiedDateTime", "subject_type"], axis=1)

['STATEFP', 'PLACEFP', 'PLACENS', 'GEOID', 'NAME', 'NAMELSAD', 'LSAD', 'CLASSFP', 'PCICBSA', 'PCINECTA', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry']


In [ ]:
# Remapping lat/lon to NA zipcodes
# Since we have more lat/lon data than zipcode data
geo_df = df[(df["Longitude"].notna()) & (df["Latitude"].notna()) & (df["Zip"].isna())]

# convert points into geodataframe
points = gpd.GeoDataFrame(geo_df[["Latitude, Longitude"]], geometry=gpd.points_from_xy(geo_df["Longitude"], geo_df["Latitude"]), crs="EPSG:4326")

zcta = zcta.to_crs(points.crs)

joined = gpd.sjoin(points, )

In [35]:
# Splitting data into test and train
# stratify is used to ensure the proportion of searches is the same in the test and train data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=10, stratify=df["search_occurred"])

In [36]:
test_df

,FieldInterviewID,NOPD_Item,District,Zone,OfficerAssignment,StopDescription,VehicleYear,VehicleMake,VehicleModel,VehicleStyle,...,search_occurred,event_dt,hour,night,month,year,weekend,isDriver,isLA,isTrafficViolation
149493,690429,J1791625,3,I,3rd District,TRAFFIC VIOLATION,2015.0,TOYOTA,CAMRY,FOUR DOOR,...,0,2025-10-18 19:17:00,19,1,10,2025,1,1,1,1
3050,565715,H0342921,3,E,3rd District,TRAFFIC VIOLATION,2018.0,VOLKSWAGEN (VW),OTHER,FOUR DOOR,...,0,2021-08-03 21:08:00,21,1,8,2021,0,1,1,1
46098,524087,J0108319,8,D,8th District,TRAFFIC VIOLATION,2007.0,TOYOTA,CAMRY,FOUR DOOR,...,0,2019-10-01 17:57:00,17,0,10,2019,0,1,1,1
233687,572446,J2723021,6,J,6th District,TRAFFIC VIOLATION,2016.0,NISSAN,ALTIMA,FOUR DOOR,...,0,2021-10-22 18:04:00,18,1,10,2021,0,0,<NA>,1
28516,539733,F0090320,3,Y,3rd District,TRAFFIC VIOLATION,2009.0,NISSAN,ALTIMA,FOUR DOOR,...,0,2020-06-01 17:48:00,17,0,6,2020,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222684,584745,F0124922,8,D,8th District,TRAFFIC VIOLATION,2014.0,HYUNDAI,ELANTRA,FOUR DOOR,...,0,2022-06-03 22:08:00,22,1,6,2022,0,0,<NA>,1
170098,665626,C2011225,5,D,5th District,TRAFFIC VIOLATION,2005.0,TOYOTA,TACOMA,PICK UP,...,0,2025-03-21 13:47:00,13,0,3,2025,0,1,<NA>,1
4476,562508,G2038121,3,E,3rd District,TRAFFIC VIOLATION,2020.0,JEEP,GRAND CHEROKEE,SPORTS UTILITY,...,0,2021-07-16 23:35:00,23,1,7,2021,0,1,1,1
179136,655663,J3112824,5,M,5th District,TRAFFIC VIOLATION,2017.0,HONDA,PILOT,FOUR DOOR,...,0,2024-10-31 01:27:00,1,1,10,2024,0,1,1,1


In [37]:
train_df.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_train1.csv", index=False)
test_df.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_test1.csv", index=False)

In [25]:
print(sum(test_df["search_occurred"]))
print(sum(train_df["search_occurred"]))

1670
6681
